# parameter recovery

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
os.environ["OMP_NUM_THREADS"] = "1"
import seaborn as sns 
from scipy.io import loadmat
import ast
from scipy.io import loadmat, savemat
import warnings
warnings.filterwarnings("ignore")
import statsmodels.api as sm


In [ ]:
outputFolderName = r"param_recovery_3_param_recovery_v1"
inputFolderName = r"\\155.100.91.44\d\Data\Nill\BART_param_recovery\new_modeling\param_recovery_2_simulated_fields"

if not os.path.exists(outputFolderName):
    os.makedirs(outputFolderName)

In [ ]:
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.io import loadmat

matFiles = [f for f in os.listdir(inputFolderName) if f.endswith(".mat")]
nPatients = len(matFiles)
rows = []

for pt in range(nPatients):
# for pt in range(2):
    fileName = matFiles[pt]

    ptID = os.path.splitext(fileName)[0]
    ptID = ptID.replace("_TDdataParamRecovery", "")

    print(f"\nprocessing pt {pt+1}/{nPatients}: {ptID}")

    matFile = os.path.join(inputFolderName, fileName)
    mat = loadmat(matFile, struct_as_record=False, squeeze_me=True)

    TDdataParamRecovery = mat["TDdataParamRecovery"]

    alphas = np.asarray(TDdataParamRecovery.a, dtype=float)
    nAlpha = len(alphas)
    nTrials = int(TDdataParamRecovery.nTrials)

    # -----------------------------
    # use simulated data for recovery
    # -----------------------------
    result_raw = np.asarray(TDdataParamRecovery.resultSimulated, dtype=str)[:nTrials]
    y = np.array([1 if x == "banked" else 0 for x in result_raw], dtype=float)

    # build reward exactly like fitted code
    if hasattr(TDdataParamRecovery, "pointsPerTrialSimulated"):
        pointsPerTrial = np.asarray(TDdataParamRecovery.pointsPerTrialSimulated, dtype=float)[:nTrials]

        reward = np.zeros(nTrials, dtype=float)
        reward[0] = pointsPerTrial[0]

        for t in range(1, nTrials):
            if result_raw[t] == "banked":
                reward[t] = pointsPerTrial[t]
            else:
                reward[t] = 0.0

    elif hasattr(TDdataParamRecovery, "pointsPerTrial"):
        # fallback if simulated points were stored under pointsPerTrial
        pointsPerTrial = np.asarray(TDdataParamRecovery.pointsPerTrial, dtype=float)[:nTrials]

        reward = np.zeros(nTrials, dtype=float)
        reward[0] = pointsPerTrial[0]

        for t in range(1, nTrials):
            if result_raw[t] == "banked":
                reward[t] = pointsPerTrial[t]
            else:
                reward[t] = 0.0

    else:
        # fallback only if rewardSimulated was already saved correctly
        reward = np.asarray(TDdataParamRecovery.rewardSimulated, dtype=float)[:nTrials]

    # -----------------------------
    # fit grid
    # -----------------------------
    fit_score = np.full((nAlpha, nAlpha), np.nan)
    expectedReward_all = np.full((nAlpha, nAlpha, nTrials), np.nan, dtype=float)
    RewardPE_all = np.full((nAlpha, nAlpha, nTrials), np.nan, dtype=float)

    for ap in range(nAlpha - 1, -1, -1):
        for an in range(nAlpha - 1, -1, -1):

            RewardPE = np.zeros(nTrials, dtype=float)
            expectedReward = np.zeros(nTrials, dtype=float)

            for t in range(1, nTrials):
                RewardPE[t] = reward[t] - expectedReward[t - 1]

                if RewardPE[t] > 0:
                    expectedReward[t] = expectedReward[t - 1] + alphas[ap] * RewardPE[t - 1]
                elif RewardPE[t] < 0:
                    expectedReward[t] = expectedReward[t - 1] + alphas[an] * RewardPE[t - 1]
                else:
                    expectedReward[t] = expectedReward[t - 1]

            expectedReward_all[ap, an, :] = expectedReward
            RewardPE_all[ap, an, :] = RewardPE

            X = sm.add_constant(expectedReward)

            try:
                if len(np.unique(y)) < 2 or not np.all(np.isfinite(X)):
                    fit_score[ap, an] = np.nan
                else:
                    model = sm.GLM(
                        y,
                        X,
                        family=sm.families.Binomial(link=sm.families.links.Logit())
                    )
                    glm_result = model.fit()
                    fit_score[ap, an] = glm_result.params[1]
            except Exception:
                fit_score[ap, an] = np.nan

    if np.all(np.isnan(fit_score)):
        print(f"{ptID}: all fit scores are NaN, skipping")
        continue

    # -----------------------------
    # MATLAB-style alpha selection
    # -----------------------------
    positive_profile = np.nanmax(fit_score, axis=0)   # leaves an
    negative_profile = np.nanmax(fit_score, axis=1)   # leaves ap

    bestAlphaNegIdx = int(np.nanargmax(positive_profile))
    bestAlphaPosIdx = int(np.nanargmax(negative_profile))

    recoveredAlphaPos = alphas[bestAlphaPosIdx]
    recoveredAlphaNeg = alphas[bestAlphaNegIdx]

    print(f"{ptID}: recoveredAlphaPos = {recoveredAlphaPos}")
    print(f"{ptID}: recoveredAlphaNeg = {recoveredAlphaNeg}")

    # -----------------------------
    # compare recovered vs simulated true params
    # -----------------------------
    rows.append({
        "ptID": ptID,
        "sim_alpha_plus": float(TDdataParamRecovery.bestAlphaPos),
        "sim_alpha_minus": float(TDdataParamRecovery.bestAlphaNeg),
        "fit_alpha_plus": recoveredAlphaPos,
        "fit_alpha_minus": recoveredAlphaNeg
    })

# -----------------------------
# save to csv
# -----------------------------
df_alpha_compare = pd.DataFrame(rows)

output_csv = os.path.join(outputFolderName, "alpha_comparison.csv")
df_alpha_compare.to_csv(output_csv, index=False)



# debug

In [ ]:
fields = [f for f in dir(TDdataParamRecovery) if not f.startswith('_')]
print(fields)